In [1]:
from pyspark.sql import SparkSession

# Création de la session Spark
spark = SparkSession.builder \
    .appName("TP_RDD_Operations") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark est prêt ! Version :", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/11 09:42:20 WARN Utils: Your hostname, codespaces-52671d, resolves to a loopback address: 127.0.0.1; using 10.0.1.246 instead (on interface eth0)
26/04/11 09:42:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/11 09:42:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark est prêt ! Version : 4.1.1


In [2]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../generate_data/synthetic_jobs_fast.csv")

In [4]:
# Number of rows
num_rows = df.count()

# Number of columns
num_cols = len(df.columns)

print(f"The dataset contains {num_rows} lines and {num_cols} columns")

The dataset contains 3000 lines and 17 columns


In [5]:
df.show(5, truncate=False)

+---+-------------+---------+--------+-----------+----------+--------+----------+----------+--------+---------+----------------+--------+---------+--------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id |title        |company  |location|date_posted|job_type  |interval|min_amount|max_amount|currency|is_remote|company_industry|city    |seniority|year_exp|education|description                                                                                                                                                                                                                                                                                                        |
+---+-------------+---------+--------+-----------+

In [6]:
import re 
def top_k_words(df, column, k):
    # 1) MAP: extract text column
    rdd_lignes = df.select(column).rdd.map(lambda row: row[0])

    # 2) MAP: tokenize
    rdd_mots = rdd_lignes.flatMap(lambda ligne: ligne.split())

    # 3) MAP: clean words
    rdd_mots_propres = rdd_mots.map(
        lambda mot: re.sub(r"[^a-zA-Z0-9]", "", mot).lower()
    )

    # 4) FILTER: remove empty + short words
    rdd_mots_filtres = rdd_mots_propres.filter(lambda mot: mot != "" and len(mot) > 1)

    # 5) MAP: (word, 1)
    rdd_paires = rdd_mots_filtres.map(lambda mot: (mot, 1))

    # 6) REDUCE: count occurrences
    rdd_comptes = rdd_paires.reduceByKey(lambda a, b: a + b)

    # 7) TOP-K (MapReduce-style sorting step)
    top_k = rdd_comptes.takeOrdered(k, key=lambda x: -x[1])

    return top_k

In [7]:
top10 = top_k_words(df, "description", 10)

print("Top 10 words:")
print(top10)

Top 10 words:
[('and', 8190), ('data', 4287), ('to', 4060), ('in', 3062), ('at', 2850), ('with', 2412), ('as', 1902), ('model', 1819), ('team', 1521), ('the', 1423)]


In [8]:
top5 = top_k_words(df, "description", 5)

print("Top 5 words:")
print(top5)

Top 5 words:
[('and', 8190), ('data', 4287), ('to', 4060), ('in', 3062), ('at', 2850)]
